# Embeddings를 활용한 Tool Search: Claude를 수천 개의 Tool로 확장하기

수십 개의 특화된 tool을 사용하는 Claude 애플리케이션을 구축하다 보면 곧 한계에 부딪힙니다. 모든 tool 정의를 처음부터 제공하면 context window를 소모하고, latency와 비용이 증가하며, Claude가 적절한 tool을 찾기 어려워집니다. 약 100개를 넘는 tool에는 이 접근법을 적용하기 어렵습니다.

Semantic tool search는 tool을 검색 가능한 리소스로 취급하여 이 문제를 해결합니다. 수백 개의 정의를 처음부터 넣는 대신, 필요에 따라 관련 기능을 반환하는 단일 `tool_search` tool을 Claude에 제공합니다. 그러면 context 사용량을 90% 이상 줄이면서 수천 개의 tool로 확장 가능한 애플리케이션을 구현할 수 있습니다.

**이 cookbook을 마치면 다음을 할 수 있습니다.**
- client-side tool search를 구현하여 Claude 애플리케이션을 수십 개에서 수천 개의 tool로 확장하기
- semantic embeddings로 작업 context에 따라 관련 tool을 동적으로 발견하기
- 도메인별 tool 라이브러리(API, database, 내부 시스템)에 이 패턴 적용하기

이 패턴은 context 효율성이 중요한 대규모 tool 생태계를 관리하는 팀들이 production에서 사용합니다. 명확한 설명을 위해 소규모 tool 집합으로 시연하지만, 같은 접근법을 수백 또는 수천 개의 tool을 포함하는 라이브러리에도 그대로 확장할 수 있습니다.

## 사전 요구사항

이 guide를 따라 하기 전에 다음을 준비하세요.

**필요한 지식**
- Python 기초 - 함수, dictionary, 기본 자료구조를 익숙하게 사용할 수 있어야 합니다.
- Claude tool use에 대한 기본 이해 - 먼저 [Tool Use Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)를 읽는 것을 권장합니다.

**필요한 도구**
- Python 3.11 이상
- Anthropic API key ([여기에서 발급](https://docs.anthropic.com/claude/reference/getting-started-with-the-api))

## 설정

먼저 필요한 dependencies를 설치합니다.


In [68]:
# 참고: stdout에 너무 많이 출력하지 않도록 -q 사용
# pythran build 문제를 피하도록 --only-binary 사용
%pip install --only-binary :all: -q anthropic sentence-transformers numpy python-dotenv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


`.env` 파일에 다음이 포함되어 있는지 확인합니다.
```
ANTHROPIC_API_KEY=your_key_here
```

환경 변수를 불러오고 client를 설정합니다.


In [69]:
import json
import random
from datetime import datetime, timedelta
from typing import Any

import anthropic
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# .env 파일에서 환경 변수 불러오기
load_dotenv()

# 쉽게 업데이트하도록 모델 constant 정의
MODEL = "claude-sonnet-4-6"

# Claude client 초기화(환경에서 API key 불러옴)
claude_client = anthropic.Anthropic()

# SentenceTransformer 모델 불러오기
# all-MiniLM-L6-v2는 384 dimensional embeddings를 사용하는 가벼운 모델입니다
# 최초 사용 시 HuggingFace에서 다운로드됩니다
print("Loading SentenceTransformer model...")
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("✓ Clients initialized successfully")

Loading SentenceTransformer model...
✓ Clients initialized successfully


## Tool 라이브러리 정의

Semantic search를 구현하려면 먼저 검색할 tool이 필요합니다. Weather와 Finance 두 범주에 걸친 8개의 tool 라이브러리를 만들겠습니다.

Production 애플리케이션에서는 내부 API, database 작업, third-party 통합에 걸쳐 수백 또는 수천 개의 tool을 관리할 수 있습니다. Semantic search 접근법은 수정 없이 이러한 대규모 라이브러리로 확장됩니다. 여기서는 시연을 명확하게 하기 위해서만 작은 집합을 사용합니다.


In [70]:
# 두 domain으로 tool 라이브러리 정의
TOOL_LIBRARY = [
    # 날씨 Tool
    {
        "name": "get_weather",
        "description": "Get the current weather in a given location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "The unit of temperature",
                },
            },
            "required": ["location"],
        },
    },
    {
        "name": "get_forecast",
        "description": "Get the weather forecast for multiple days ahead",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state",
                },
                "days": {
                    "type": "number",
                    "description": "Number of days to forecast (1-10)",
                },
            },
            "required": ["location", "days"],
        },
    },
    {
        "name": "get_timezone",
        "description": "Get the current timezone and time for a location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name or timezone identifier",
                }
            },
            "required": ["location"],
        },
    },
    {
        "name": "get_air_quality",
        "description": "Get current air quality index and pollutant levels for a location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name or coordinates",
                }
            },
            "required": ["location"],
        },
    },
    # Finance Tool
    {
        "name": "get_stock_price",
        "description": "Get the current stock price and market data for a given ticker symbol",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol (e.g., AAPL, GOOGL)",
                },
                "include_history": {
                    "type": "boolean",
                    "description": "Include historical data",
                },
            },
            "required": ["ticker"],
        },
    },
    {
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another using current exchange rates",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "Amount to convert",
                },
                "from_currency": {
                    "type": "string",
                    "description": "Source currency code (e.g., USD)",
                },
                "to_currency": {
                    "type": "string",
                    "description": "Target currency code (e.g., EUR)",
                },
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
    {
        "name": "calculate_compound_interest",
        "description": "Calculate compound interest for investments over time",
        "input_schema": {
            "type": "object",
            "properties": {
                "principal": {
                    "type": "number",
                    "description": "Initial investment amount",
                },
                "rate": {
                    "type": "number",
                    "description": "Annual interest rate (as percentage)",
                },
                "years": {"type": "number", "description": "Number of years"},
                "frequency": {
                    "type": "string",
                    "enum": ["daily", "monthly", "quarterly", "annually"],
                    "description": "Compounding frequency",
                },
            },
            "required": ["principal", "rate", "years"],
        },
    },
    {
        "name": "get_market_news",
        "description": "Get recent financial news and market updates for a specific company or sector",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Company name, ticker symbol, or sector",
                },
                "limit": {
                    "type": "number",
                    "description": "Maximum number of news articles to return",
                },
            },
            "required": ["query"],
        },
    },
]

print(f"✓ Defined {len(TOOL_LIBRARY)} tools in the library")

✓ Defined 8 tools in the library


## Tool Embeddings 생성

Semantic search는 단순히 keyword를 검색하는 대신 텍스트의 *의미*를 비교합니다. 이를 위해 각 tool 정의를 의미를 포착하는 **embedding vector**로 변환해야 합니다.

Tool 정의는 이름, 설명, parameter가 포함된 구조화된 JSON 객체이므로, 먼저 각 tool을 사람이 읽을 수 있는 텍스트 표현으로 변환한 다음 SentenceTransformer의 `all-MiniLM-L6-v2` 모델로 embedding vector를 생성합니다.

이 모델을 선택한 이유는 다음과 같습니다.
- **가볍고 빠릅니다.** (더 큰 모델의 768개 이상에 비해 384개 dimension만 사용)
- API 호출 없이 **로컬에서 실행됩니다.**
- **Tool search에 충분합니다.** (더 나은 정확도를 위해 더 큰 모델을 실험해 볼 수 있습니다.)

Tool 정의를 검색 가능한 텍스트로 변환하는 함수를 만드는 것부터 시작하겠습니다.


In [71]:
def tool_to_text(tool: dict[str, Any]) -> str:
    """
    Tool 정의를 embedding을 위한 텍스트 표현으로 변환합니다.
    Tool 이름, 설명, parameter 정보를 결합합니다.
    """
    text_parts = [
        f"Tool: {tool['name']}",
        f"Description: {tool['description']}",
    ]

    # Parameter 정보 추가
    if "input_schema" in tool and "properties" in tool["input_schema"]:
        params = tool["input_schema"]["properties"]
        param_descriptions = []
        for param_name, param_info in params.items():
            param_desc = param_info.get("description", "")
            param_type = param_info.get("type", "")
            param_descriptions.append(f"{param_name} ({param_type}): {param_desc}")

        if param_descriptions:
            text_parts.append("Parameters: " + ", ".join(param_descriptions))

    return "\n".join(text_parts)


# Tool 하나로 테스트
sample_text = tool_to_text(TOOL_LIBRARY[0])
print("Sample tool text representation:")
print(sample_text)

Sample tool text representation:
Tool: get_weather
Description: Get the current weather in a given location
Parameters: location (string): The city and state, e.g. San Francisco, CA, unit (string): The unit of temperature


이제 모든 tool의 embeddings를 생성하겠습니다.


In [72]:
# 모든 tool의 embeddings 생성
print("Creating embeddings for all tools...")

tool_texts = [tool_to_text(tool) for tool in TOOL_LIBRARY]

# SentenceTransformer로 모든 tool을 한 번에 embedding
# 모델은 기본적으로 normalized embeddings를 반환합니다
tool_embeddings = embedding_model.encode(tool_texts, convert_to_numpy=True)

print(f"✓ Created embeddings with shape: {tool_embeddings.shape}")
print(f"  - {tool_embeddings.shape[0]} tools")
print(f"  - {tool_embeddings.shape[1]} dimensions per embedding")

Creating embeddings for all tools...
✓ Created embeddings with shape: (8, 384)
  - 8 tools
  - 384 dimensions per embedding


## Tool Search 구현

Tool을 vector로 embedding했으므로 이제 semantic search를 구현할 수 있습니다. 두 텍스트의 의미가 비슷하면 embedding vector도 vector space에서 서로 가깝습니다. 이 "가까움"을 **cosine similarity**로 측정합니다.

검색 과정은 다음과 같습니다.
1. **Query embedding**: Claude의 자연어 검색 요청을 tool과 같은 vector space로 변환합니다.
2. **유사도 계산**: query vector와 각 tool vector 사이의 cosine similarity를 계산합니다.
3. **순위 지정 및 반환**: 유사도 점수로 tool을 정렬하여 상위 N개 일치 항목을 반환합니다.

Semantic search를 사용하면 Claude는 정확한 tool 이름 대신 "날씨를 확인해야 해"나 "투자 수익 계산" 같은 자연어로 검색할 수 있습니다.

검색 함수를 구현하고 예제 query로 테스트하겠습니다.


In [73]:
def search_tools(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    """
    Semantic similarity로 tool을 검색합니다.

    Args:
        query: 필요한 tool에 대한 자연어 설명
        top_k: 반환할 상위 tool 수

    Returns:
        Query에 가장 관련된 tool 정의 목록
    """
    # SentenceTransformer로 query embedding
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)

    # Dot product로 cosine similarity 계산
    # SentenceTransformer는 normalized embeddings를 반환하므로 dot product = cosine similarity
    similarities = np.dot(tool_embeddings, query_embedding)

    # 상위 k개의 index 가져오기
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    # 해당 tool과 점수 반환
    results = []
    for idx in top_indices:
        results.append({"tool": TOOL_LIBRARY[idx], "similarity_score": float(similarities[idx])})

    return results


# 검색 함수 테스트
test_query = "I need to check the weather"
test_results = search_tools(test_query, top_k=3)

print(f"Search query: '{test_query}'\n")
print("Top 3 matching tools:")
for i, result in enumerate(test_results, 1):
    tool_name = result["tool"]["name"]
    score = result["similarity_score"]
    print(f"{i}. {tool_name} (similarity: {score:.3f})")

Search query: 'I need to check the weather'

Top 3 matching tools:
1. get_weather (similarity: 0.560)
2. get_forecast (similarity: 0.508)
3. get_air_quality (similarity: 0.401)


## tool_search Tool 정의

이제 Claude가 필요에 따라 다른 tool을 발견할 수 있게 하는 **meta-tool**을 구현하겠습니다. Claude는 자신에게 없는 기능이 필요할 때 이 `tool_search` tool로 검색하여 결과에서 tool 정의를 받고, 새로 발견한 tool을 즉시 사용할 수 있습니다.

처음에 Claude에 제공하는 tool은 이것 하나뿐입니다.


In [74]:
# tool_search tool 정의
TOOL_SEARCH_DEFINITION = {
    "name": "tool_search",
    "description": "Search for available tools that can help with a task. Returns tool definitions for matching tools. Use this when you need a tool but don't have it available yet.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Natural language description of what kind of tool you need (e.g., 'weather information', 'currency conversion', 'stock prices')",
            },
            "top_k": {
                "type": "number",
                "description": "Number of tools to return (default: 5)",
            },
        },
        "required": ["query"],
    },
}

print("✓ Tool search definition created")

✓ Tool search definition created


이제 Claude의 `tool_search` 호출을 처리하고 발견한 tool을 반환하는 handler를 구현하겠습니다.


In [75]:
def handle_tool_search(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    """
    tool_search 호출을 처리하고 tool 참조를 반환합니다.

    발견한 tool의 tool_reference content block 목록을 반환합니다.
    """
    # 관련 tool 검색
    results = search_tools(query, top_k=top_k)

    # 전체 정의 대신 tool_reference 객체 생성
    tool_references = [
        {"type": "tool_reference", "tool_name": result["tool"]["name"]} for result in results
    ]

    print(f"\n🔍 Tool search: '{query}'")
    print(f"   Found {len(tool_references)} tools:")
    for i, result in enumerate(results, 1):
        print(f"   {i}. {result['tool']['name']} (similarity: {result['similarity_score']:.3f})")

    return tool_references


# Handler 테스트
test_result = handle_tool_search("stock market data", top_k=3)
print(f"\nReturned {len(test_result)} tool references:")
for ref in test_result:
    print(f"  {ref}")


🔍 Tool search: 'stock market data'
   Found 3 tools:
   1. get_stock_price (similarity: 0.524)
   2. get_market_news (similarity: 0.469)
   3. calculate_compound_interest (similarity: 0.244)

Returned 3 tool references:
  {'type': 'tool_reference', 'tool_name': 'get_stock_price'}
  {'type': 'tool_reference', 'tool_name': 'get_market_news'}
  {'type': 'tool_reference', 'tool_name': 'calculate_compound_interest'}


## Mock Tool 실행

이 시연에서는 tool 실행에 대한 mock 응답을 만들겠습니다. 실제 애플리케이션에서는 실제 API나 서비스를 호출합니다.


In [76]:
def mock_tool_execution(tool_name: str, tool_input: dict[str, Any]) -> str:
    """
    Tool 실행에 대한 현실적인 mock 응답을 생성합니다.

    Args:
        tool_name: 실행하는 tool의 이름
        tool_input: Tool의 input parameter

    Returns:
        Tool에 적합한 mock 응답 문자열
    """
    # 날씨 tool
    if tool_name == "get_weather":
        location = tool_input.get("location", "Unknown")
        unit = tool_input.get("unit", "fahrenheit")
        temp = random.randint(15, 30) if unit == "celsius" else random.randint(60, 85)
        conditions = random.choice(["sunny", "partly cloudy", "cloudy", "rainy"])
        return json.dumps(
            {
                "location": location,
                "temperature": temp,
                "unit": unit,
                "conditions": conditions,
                "humidity": random.randint(40, 80),
                "wind_speed": random.randint(5, 20),
            }
        )

    elif tool_name == "get_forecast":
        location = tool_input.get("location", "Unknown")
        days = int(tool_input.get("days", 5))
        forecast = []
        for i in range(days):
            date = (datetime.now() + timedelta(days=i)).strftime("%Y-%m-%d")
            forecast.append(
                {
                    "date": date,
                    "high": random.randint(20, 30),
                    "low": random.randint(10, 20),
                    "conditions": random.choice(["sunny", "cloudy", "rainy", "partly cloudy"]),
                }
            )
        return json.dumps({"location": location, "forecast": forecast})

    elif tool_name == "get_timezone":
        location = tool_input.get("location", "Unknown")
        return json.dumps(
            {
                "location": location,
                "timezone": "UTC+9",
                "current_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "utc_offset": "+09:00",
            }
        )

    elif tool_name == "get_air_quality":
        location = tool_input.get("location", "Unknown")
        aqi = random.randint(20, 150)
        categories = {
            (0, 50): "Good",
            (51, 100): "Moderate",
            (101, 150): "Unhealthy for Sensitive Groups",
        }
        category = next(cat for (low, high), cat in categories.items() if low <= aqi <= high)
        return json.dumps(
            {
                "location": location,
                "aqi": aqi,
                "category": category,
                "pollutants": {
                    "pm25": random.randint(5, 50),
                    "pm10": random.randint(10, 100),
                    "o3": random.randint(20, 80),
                },
            }
        )

    # Finance tool
    elif tool_name == "get_stock_price":
        ticker = tool_input.get("ticker", "UNKNOWN")
        return json.dumps(
            {
                "ticker": ticker,
                "price": round(random.uniform(100, 500), 2),
                "change": round(random.uniform(-5, 5), 2),
                "change_percent": round(random.uniform(-2, 2), 2),
                "volume": random.randint(1000000, 10000000),
                "market_cap": f"${random.randint(100, 1000)}B",
            }
        )

    elif tool_name == "convert_currency":
        amount = tool_input.get("amount", 0)
        from_currency = tool_input.get("from_currency", "USD")
        to_currency = tool_input.get("to_currency", "EUR")
        # Mock 환율
        rate = random.uniform(0.8, 1.2)
        converted = round(amount * rate, 2)
        return json.dumps(
            {
                "original_amount": amount,
                "from_currency": from_currency,
                "to_currency": to_currency,
                "exchange_rate": round(rate, 4),
                "converted_amount": converted,
            }
        )

    elif tool_name == "calculate_compound_interest":
        principal = tool_input.get("principal", 0)
        rate = tool_input.get("rate", 0)
        years = tool_input.get("years", 0)
        frequency = tool_input.get("frequency", "monthly")

        # 복리 계산
        n_map = {"daily": 365, "monthly": 12, "quarterly": 4, "annually": 1}
        n = n_map.get(frequency, 12)
        final_amount = principal * (1 + rate / 100 / n) ** (n * years)
        interest_earned = final_amount - principal

        return json.dumps(
            {
                "principal": principal,
                "rate": rate,
                "years": years,
                "compounding_frequency": frequency,
                "final_amount": round(final_amount, 2),
                "interest_earned": round(interest_earned, 2),
            }
        )

    elif tool_name == "get_market_news":
        query = tool_input.get("query", "")
        limit = tool_input.get("limit", 5)
        news = []
        for i in range(min(limit, 5)):
            news.append(
                {
                    "title": f"{query} - News Article {i + 1}",
                    "source": random.choice(
                        [
                            "Bloomberg",
                            "Reuters",
                            "Financial Times",
                            "Wall Street Journal",
                        ]
                    ),
                    "published": (datetime.now() - timedelta(hours=random.randint(1, 24))).strftime(
                        "%Y-%m-%d %H:%M"
                    ),
                    "summary": f"Latest developments regarding {query}...",
                }
            )
        return json.dumps({"query": query, "articles": news, "count": len(news)})

    # 기본 fallback
    else:
        return json.dumps(
            {
                "status": "executed",
                "tool": tool_name,
                "message": f"Tool {tool_name} executed successfully with input: {json.dumps(tool_input)}",
            }
        )


print("✓ Mock tool execution function created")

✓ Mock tool execution function created


## 대화 Loop 구현

이제 모두 연결해 보겠습니다! 전체 tool search workflow를 처리하는 대화 loop를 만들겠습니다.

**대화 흐름:**
1. Claude는 `tool_search` tool만 사용 가능한 상태로 시작합니다.
2. Claude가 `tool_search`를 호출하면 semantic search를 실행하고 일치하는 tool 정의를 반환합니다.
3. 그러면 Claude는 발견한 tool을 즉시 사용할 수 있습니다.
4. Claude가 발견한 tool을 호출하면 이를 실행합니다. (이 demo에서는 mock 응답 사용)
5. Claude가 최종 답변을 얻을 때까지 loop를 계속합니다.


In [77]:
def run_tool_search_conversation(user_message: str, max_turns: int = 5) -> None:
    """
    Tool search 패턴으로 Claude와 대화합니다.

    Args:
        user_message: 최초 user message
        max_turns: 최대 대화 turn 수
    """
    print(f"\n{'=' * 80}")
    print(f"USER: {user_message}")
    print(f"{'=' * 80}\n")

    # tool_search만 사용 가능한 대화 초기화
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        print(f"\n--- Turn {turn + 1} ---")

        # 현재 message 기록으로 Claude 호출
        response = claude_client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOL_LIBRARY + [TOOL_SEARCH_DEFINITION],
            messages=messages,
            # 중요: 이 beta header는 tool 결과의 tool 정의를 활성화합니다
            extra_headers={"anthropic-beta": "advanced-tool-use-2025-11-20"},
        )

        # Assistant 응답을 messages에 추가
        messages.append({"role": "assistant", "content": response.content})

        # 완료 여부 확인
        if response.stop_reason == "end_turn":
            print("\n✓ Conversation complete\n")
            # 최종 응답 출력
            for block in response.content:
                if block.type == "text":
                    print(f"ASSISTANT: {block.text}")
            break

        # Tool use 처리
        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "text":
                    print(f"\nASSISTANT: {block.text}")

                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input
                    tool_use_id = block.id

                    print(f"\n🔧 Tool invocation: {tool_name}")
                    print(f"   Input: {json.dumps(tool_input, indent=2)}")

                    if tool_name == "tool_search":
                        # Tool search 처리
                        query = tool_input["query"]
                        top_k = tool_input.get("top_k", 5)

                        # Tool 참조 가져오기
                        tool_references = handle_tool_search(query, top_k)

                        # tool_reference content block으로 tool 결과 생성
                        tool_results.append(
                            {
                                "type": "tool_result",
                                "tool_use_id": tool_use_id,
                                "content": tool_references,
                            }
                        )
                    else:
                        # Mock 데이터로 발견한 tool 실행
                        mock_result = mock_tool_execution(tool_name, tool_input)

                        # 결과 미리보기 출력
                        if len(mock_result) > 150:
                            print(f"   ✅ Mock result: {mock_result[:150]}...")
                        else:
                            print(f"   ✅ Mock result: {mock_result}")

                        tool_results.append(
                            {
                                "type": "tool_result",
                                "tool_use_id": tool_use_id,
                                "content": mock_result,
                            }
                        )

            # Messages에 tool 결과 추가
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
        else:
            print(f"\nUnexpected stop reason: {response.stop_reason}")
            break

    print(f"\n{'=' * 80}\n")


print("✓ Conversation loop implemented")

✓ Conversation loop implemented


## 예제 1: 날씨 Query

간단한 날씨 질문으로 테스트하겠습니다. Claude는 다음을 수행해야 합니다.
1. 날씨 tool을 찾기 위해 `tool_search` 호출
2. 결과에서 날씨 tool 정의 받기
3. 발견한 tool 중 하나 사용


In [78]:
run_tool_search_conversation("What's the weather like in Tokyo?")


USER: What's the weather like in Tokyo?


--- Turn 1 ---

🔧 Tool invocation: get_weather
   Input: {
  "location": "Tokyo"
}
   ✅ Mock result: {"location": "Tokyo", "temperature": 75, "unit": "fahrenheit", "conditions": "partly cloudy", "humidity": 61, "wind_speed": 9}

--- Turn 2 ---

✓ Conversation complete

ASSISTANT: The weather in Tokyo is currently:
- **Temperature:** 75°F (about 24°C)
- **Conditions:** Partly cloudy
- **Humidity:** 61%
- **Wind Speed:** 9 mph

It's a pleasant day with comfortable temperatures and some cloud cover!




## 예제 2: Finance Query

Finance tool을 발견하여 사용해야 하는 금융 계산 query를 시도하겠습니다.


In [79]:
run_tool_search_conversation(
    "If I invest $10,000 at 5% annual interest for 10 years with monthly compounding, how much will I have?"
)


USER: If I invest $10,000 at 5% annual interest for 10 years with monthly compounding, how much will I have?


--- Turn 1 ---

🔧 Tool invocation: calculate_compound_interest
   Input: {
  "principal": 10000,
  "rate": 5,
  "years": 10,
  "frequency": "monthly"
}
   ✅ Mock result: {"principal": 10000, "rate": 5, "years": 10, "compounding_frequency": "monthly", "final_amount": 16470.09, "interest_earned": 6470.09}

--- Turn 2 ---

✓ Conversation complete

ASSISTANT: If you invest $10,000 at 5% annual interest for 10 years with monthly compounding, you will have:

**Final Amount: $16,470.09**

This means you'll earn **$6,470.09** in interest over the 10-year period.

The monthly compounding means that interest is calculated and added to your principal every month, which allows your investment to grow faster than with annual compounding due to the effect of earning "interest on interest" more frequently.




## 결론

이 cookbook에서는 Claude가 대규모 tool 라이브러리를 효율적으로 다룰 수 있도록 client-side tool search 시스템을 구현했습니다. 다음을 다루었습니다.

- **Semantic tool discovery**: embeddings를 사용하여 자연어 query를 관련 tool과 연결함으로써, 사용 가능한 모든 tool을 처음부터 보지 않고도 Claude가 적절한 기능을 찾도록 합니다.
- **동적 tool loading**: Claude의 tool search 기능으로 tool 결과에 tool 정의를 반환하여, 대화 중에 Claude가 새로운 tool을 발견하고 즉시 사용하도록 합니다.
- **Context 최적화**: 초기 context를 수천 token(19개 이상의 tool 정의)에서 단일 `tool_search` 정의로 줄여 context 사용량을 90% 이상 절감합니다.

### 프로젝트에 적용하기

다음과 같은 경우 tool search를 고려하세요.
- **특화된 tool이 20개를 넘고** context 사용량이 문제가 되는 경우
- Tool 라이브러리가 **시간에 따라 커져** 수동으로 선별하기 어려운 경우
- 수백 개의 endpoint가 있는 **도메인별 API**를 지원해야 하는 경우(database 작업, 내부 microservice, third-party 통합)
- 애플리케이션에서 **비용 및 latency 최적화**가 우선순위인 경우

### 다음 단계

이 구현을 더 발전시키려면 다음을 수행하세요.

1. **Embeddings 저장**: 매 session마다 다시 계산하지 않도록 embeddings를 disk에 cache하여 시작 시간을 줄입니다.
2. **검색 품질 개선**: 다른 embedding 모델(예: `all-mpnet-base-v2`와 같은 더 큰 모델)을 실험하거나 semantic matching과 keyword matching(BM25)을 결합하는 hybrid search를 구현합니다.
3. **더 큰 라이브러리로 확장**: 수백 또는 수천 개의 tool로 테스트하여 production 규모에서 이 패턴이 어떻게 작동하는지 확인합니다.
4. **Tool metadata 추가**: 검색 순위에 사용 통계, 비용 정보, 신뢰성 점수를 포함합니다.
5. **Caching 구현**: 자주 사용하는 tool 정의를 cache하여 반복 검색을 줄입니다.

### 추가 자료

- [Claude Tool Use Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use) - tool을 활용한 구축에 관한 종합 guide
- [SentenceTransformers Documentation](https://www.sbert.net/) - embedding 모델과 semantic search에 대해 더 알아보기
- [Tool Search Tool Documentation](https://docs.anthropic.com/en/docs/build-with-claude/tool-use#tool-search) - tool search 패턴에 관한 공식 문서
